# Hypothesis 4 — Does the Test group spend less time per step?

### Step 1 — Define the hypothesis
Here we write the two statements we want to test.

- **Null hypothesis (H0):** The Test group does **not** spend less time per step than the Control group.
- **Alternative hypothesis (H1):** The Test group **does** spend less time per step than the Control group.

In [39]:
# We write the hypothesis as simple text so it is easy to present
H0_4 = 'The Test group does not spend less time per step than the Control group.'
H1_4 = 'The Test group spends less time per step than the Control group.'

print('H0:', H0_4)
print('H1:', H1_4)

H0: The Test group does not spend less time per step than the Control group.
H1: The Test group spends less time per step than the Control group.


### Step 2 — Build the time-per-step dataset
We need to calculate how long each client spent at each step.

The approach:
1. Sort events by client and time
2. Calculate the time difference between consecutive steps for each client
3. Keep only valid transitions (same client, forward in time)

In [40]:
# Corre esto y pégame el output
import pandas as pd

# ¿Qué variables tienes cargadas?
print(dir())

['H0_4', 'H1_4', 'In', 'Out', '_', '__', '___', '__builtin__', '__builtins__', '__doc__', '__loader__', '__name__', '__package__', '__spec__', '__vsc_ipynb_file__', '_dh', '_i', '_i1', '_i10', '_i11', '_i12', '_i13', '_i14', '_i15', '_i16', '_i17', '_i18', '_i19', '_i2', '_i20', '_i21', '_i22', '_i23', '_i24', '_i25', '_i26', '_i27', '_i28', '_i29', '_i3', '_i30', '_i31', '_i32', '_i33', '_i34', '_i35', '_i36', '_i37', '_i38', '_i39', '_i4', '_i40', '_i5', '_i6', '_i7', '_i8', '_i9', '_ih', '_ii', '_iii', '_oh', 'alpha', 'control_time', 'control_time_mean', 'decision_4', 'df', 'df_demo', 'df_experiment', 'df_final', 'df_time', 'df_web', 'df_web_1', 'df_web_2', 'dirs', 'exit', 'f', 'files', 'get_ipython', 'open', 'os', 'p_value_4', 'path', 'pd', 'quit', 'result', 'root', 'simple_result_4', 'subprocess', 't_stat_4', 'test_time', 'test_time_mean', 'time_difference', 'ttest_ind', 'variation_col']


In [41]:
import os

# Mira qué archivos CSV/Excel tienes en tu carpeta de trabajo
for f in os.listdir('.'):
    if f.endswith(('.csv', '.xlsx', '.parquet', '.json')):
        print(f)

we_fn_use_c_marketing_customer_value_analysis.csv


In [42]:
import pandas as pd

df = pd.read_csv('we_fn_use_c_marketing_customer_value_analysis.csv')
print(df.shape)
print(df.columns.tolist())
print(df.head(3))

(9134, 24)
['Customer', 'State', 'Customer Lifetime Value', 'Response', 'Coverage', 'Education', 'Effective To Date', 'EmploymentStatus', 'Gender', 'Income', 'Location Code', 'Marital Status', 'Monthly Premium Auto', 'Months Since Last Claim', 'Months Since Policy Inception', 'Number of Open Complaints', 'Number of Policies', 'Policy Type', 'Policy', 'Renew Offer Type', 'Sales Channel', 'Total Claim Amount', 'Vehicle Class', 'Vehicle Size']
  Customer       State  Customer Lifetime Value Response  Coverage Education  \
0  BU79786  Washington              2763.519279       No     Basic  Bachelor   
1  QZ44356     Arizona              6979.535903       No  Extended  Bachelor   
2  AI49188      Nevada             12887.431650       No   Premium  Bachelor   

  Effective To Date EmploymentStatus Gender  Income  ...  \
0           2/24/11         Employed      F   56274  ...   
1           1/31/11       Unemployed      F       0  ...   
2           2/19/11         Employed      F   48767  .

In [43]:
import os

path = r'C:\Users\User\Desktop\GitHub\lab-dw-pandas\Ironhack_second_project\raw'

for f in os.listdir(path):
    print(f)

.gitignore
df_final_demo.csv
df_final_experiment_clients.csv
df_final_web_data_pt_1.csv
df_final_web_data_pt_2.csv


In [44]:
import os

for root, dirs, files in os.walk('C:/Users/User'):
    for f in files:
        if f.endswith('.csv'):
            print(os.path.join(root, f))
    # Limitar profundidad
    if root.count(os.sep) > 5:
        break

In [45]:
import pandas as pd

path = r'C:\Users\User\Desktop\GitHub\lab-dw-pandas\Ironhack_second_project\raw'

# Cargar los 4 archivos
df_demo        = pd.read_csv(f'{path}/df_final_demo.csv')
df_experiment  = pd.read_csv(f'{path}/df_final_experiment_clients.csv')
df_web_1       = pd.read_csv(f'{path}/df_final_web_data_pt_1.csv')
df_web_2       = pd.read_csv(f'{path}/df_final_web_data_pt_2.csv')

# Unir los dos archivos de web data
df_web = pd.concat([df_web_1, df_web_2], ignore_index=True)

# Unir con el grupo de experimento (Test / Control)
df_final = df_web.merge(df_experiment, on='client_id', how='inner')

# Convertir date_time a datetime
df_final['date_time'] = pd.to_datetime(df_final['date_time'])

# Nombre de la columna de variación
variation_col = 'Variation'

print('df_final shape:', df_final.shape)
print('Columns:', df_final.columns.tolist())
print(df_final.head(3))

df_final shape: (449831, 6)
Columns: ['client_id', 'visitor_id', 'visit_id', 'process_step', 'date_time', 'Variation']
   client_id            visitor_id                      visit_id process_step  \
0    9988021  580560515_7732621733  781255054_21935453173_531117       step_3   
1    9988021  580560515_7732621733  781255054_21935453173_531117       step_2   
2    9988021  580560515_7732621733  781255054_21935453173_531117       step_3   

            date_time Variation  
0 2017-04-17 15:27:07      Test  
1 2017-04-17 15:26:51      Test  
2 2017-04-17 15:19:22      Test  


In [46]:
# Sort by client and datetime so we can measure time between steps
df_time = df_final.sort_values(['client_id', 'date_time']).copy()

# Calculate time difference in seconds between consecutive rows per client
df_time['time_spent'] = (
    df_time.groupby('client_id')['date_time']
    .diff()
    .dt.total_seconds()
)

# Remove the first event per client (no previous step to compare to)
df_time = df_time.dropna(subset=['time_spent']).copy()

# Remove negative or zero time (data issues)
df_time = df_time[df_time['time_spent'] > 0].copy()

print('Rows with valid time_spent:', df_time.shape[0])
print(df_time[['client_id', 'Variation', 'process_step', 'time_spent']].head())

Rows with valid time_spent: 372563
        client_id Variation process_step  time_spent
280825        169       NaN       step_1         9.0
280824        169       NaN       step_2        46.0
280823        169       NaN       step_3        94.0
280822        169       NaN      confirm        64.0
70802         555      Test       step_1         7.0


In [47]:
# Sort by client and datetime so we can measure time between steps
df_time = df_final.sort_values(['client_id', 'date_time']).copy()

# Calculate time difference in seconds between consecutive rows per client
df_time['time_spent'] = (
    df_time.groupby('client_id')['date_time']
    .diff()
    .dt.total_seconds()
)

# Remove the first event per client (no previous step to compare to)
df_time = df_time.dropna(subset=['time_spent']).copy()

# Remove negative or zero time (data issues)
df_time = df_time[df_time['time_spent'] > 0].copy()

print('Rows with valid time_spent:', df_time.shape[0])
print(df_time[['client_id', variation_col, 'process_step', 'time_spent']].head())

Rows with valid time_spent: 372563
        client_id Variation process_step  time_spent
280825        169       NaN       step_1         9.0
280824        169       NaN       step_2        46.0
280823        169       NaN       step_3        94.0
280822        169       NaN      confirm        64.0
70802         555      Test       step_1         7.0


In [48]:
# Remove rows where Variation is NaN (clients not in the experiment)
df_time = df_time.dropna(subset=['Variation']).copy()

print('Rows after removing NaN Variation:', df_time.shape[0])
print(df_time['Variation'].value_counts())

Rows after removing NaN Variation: 266207
Variation
Test       149534
Control    116673
Name: count, dtype: int64


### Step 3 — Separate Test and Control groups
We split the time data into the two groups so we can compare them.

In [49]:
control_time = df_time[df_time[variation_col] == 'Control']['time_spent']
test_time    = df_time[df_time[variation_col] == 'Test']['time_spent']

print('Control group rows:', control_time.shape[0])
print('Test group rows:   ', test_time.shape[0])

Control group rows: 116673
Test group rows:    149534


### Step 4 — Calculate average time per step for each group
Before running any test, we look at the raw averages.

**Average time per step** = total seconds spent / number of step transitions

In [50]:
control_time_mean = control_time.mean()
test_time_mean    = test_time.mean()
time_difference   = test_time_mean - control_time_mean

print('Control average time per step (seconds):', round(control_time_mean, 2))
print('Test average time per step (seconds):   ', round(test_time_mean, 2))
print('Difference (Test minus Control):        ', round(time_difference, 2))

Control average time per step (seconds): 86747.14
Test average time per step (seconds):    72158.26
Difference (Test minus Control):         -14588.88


### Step 5 — Choose the test and set alpha
We use a **t-test** because we are comparing **two averages** (continuous data).

We use `alternative='less'` because our H1 says the Test group spends **less** time.

We also set **alpha = 0.05** — a 5% chance of being wrong is acceptable.

In [51]:
alpha = 0.05
print('Chosen test: Independent samples t-test (Welch)')
print('Alternative: less  (Test < Control)')
print('Alpha:', alpha)

Chosen test: Independent samples t-test (Welch)
Alternative: less  (Test < Control)
Alpha: 0.05


### Step 6 — Run the statistical test
Now we run the t-test. It gives us two main results:
- **t-statistic** — how large the difference is relative to the spread
- **p-value** — the probability of seeing this result if H0 were true

The p-value is the most important one for our decision.

In [52]:
from scipy.stats import ttest_ind

t_stat_4, p_value_4 = ttest_ind(
    test_time,
    control_time,
    equal_var=False,        # Welch's t-test — safer when group sizes differ
    alternative='less'      # H1: Test mean < Control mean
)

print('T-statistic:', round(t_stat_4, 4))
print('P-value:    ', p_value_4)

T-statistic: -7.6473
P-value:     1.0303057392507552e-14


### Step 7 — Make the decision
Decision rule:
- if p-value < 0.05, we **reject H0** → the Test group is significantly faster
- if p-value >= 0.05, we **fail to reject H0** → we do not have enough evidence

In [53]:
if p_value_4 < alpha:
    decision_4 = 'Reject H0'
    simple_result_4 = 'The Test group spends significantly less time per step than the Control group.'
else:
    decision_4 = 'Fail to reject H0'
    simple_result_4 = 'We do not have enough evidence to say the Test group is faster per step.'

print('Decision:      ', decision_4)
print('Simple result: ', simple_result_4)

Decision:       Reject H0
Simple result:  The Test group spends significantly less time per step than the Control group.


### Step 8 — Write the business explanation
This final step turns the statistical result into very simple English for presentation.

In [54]:
print('Business explanation:')
print('Control group average time per step:', round(control_time_mean, 2), 'seconds')
print('Test group average time per step:   ', round(test_time_mean, 2), 'seconds')
print('Difference:                         ', round(time_difference, 2), 'seconds')
print()
print(simple_result_4)
print('A faster experience per step may indicate the new design reduces user friction.')

Business explanation:
Control group average time per step: 86747.14 seconds
Test group average time per step:    72158.26 seconds
Difference:                          -14588.88 seconds

The Test group spends significantly less time per step than the Control group.
A faster experience per step may indicate the new design reduces user friction.
